## Dataset selection

In [1]:
import re
import pandas as pd
from pathlib import Path

In [2]:
from utils import construct_instance_log_dict
partition = "test"
dataset = "14_bounded_strongly_correlated"
INCLUDE_CONPAS = False
base_path = f"../wkdir/{dataset}/{partition}"

methods_paths = []
configurations = [
    ("thresholded_expected_error","graph_with_literals_3_GTR", "backpas"),
    # ("fixed_three_ratios","graph_with_literals_8_GTR", "backpas-net"),
    # ("thresholded_expected_error","graph_with_variables_2_GCN", "backpas-param"),
    # ("fixed_three_ratios","graph_with_variables_2_GCN", "backpas-v0"),
]

method_to_name = {}

for method,network, name in configurations:
    backpas_network_method_paths = f"{base_path}/{network}/trust_region_{method}"
    backpas_network_method_paths = Path(backpas_network_method_paths)
    methods_to_add = [method_path for method_path in backpas_network_method_paths.iterdir() if method_path.is_dir() and not method_path.name.endswith("_log")]
    methods_paths += methods_to_add 
    if len(methods_to_add) == 1:
        method_to_name[methods_to_add[0].name] = name

print(f"Methods paths: {methods_paths}")
baseline_path = Path(f"{base_path}/baseline")
conpas_path = Path(f"{base_path}/conpas_paper")



if INCLUDE_CONPAS:
    methods_paths.append(conpas_path)
instance_log_dict = construct_instance_log_dict(
    baseline_path=baseline_path,
    methods_paths=methods_paths
)

Methods paths: [PosixPath('../wkdir/14_bounded_strongly_correlated/test/graph_with_literals_3_GTR/trust_region_thresholded_expected_error/threshold_0.8005575056693743_alpha_0.4161451548840182')]
Method threshold_0.8005575056693743_alpha_0.4161451548840182 has 0 instances with timeout.


## Test hyperparameters selected

In [3]:
if dataset== "MIS" or dataset == "MVC":
    n_nodes = 6000
elif dataset == "CA":
    n_nodes = 4000
elif dataset == "MIS-mixed-train" or dataset == "14_bounded_strongly_correlated":
    n_nodes = None # No fixed number of nodes
else:
    raise ValueError(f"Dataset {dataset} not recognized.")

for key, value in method_to_name.items():
    if key.startswith("threshold"):
        key_parts = key.split("_")
        print(f"{value} : $({key_parts[1]}, {key_parts[3]})$")
    elif key.startswith("k_0"):
        key_parts = key.split("_")
        k = float(key_parts[1])
        value_0 = float(key_parts[4])
        delta = float(key_parts[6])
        Delta = int(k * n_nodes * delta)
        K_0 = int(k * n_nodes * value_0)
        K_1 = int(k * n_nodes * (1 - value_0))
        print(f"{value} : $({K_0}, {K_1}, {Delta})$")
    else:
        raise ValueError(f"Method name {key} not recognized.")

backpas : $(0.8005575056693743, 0.4161451548840182)$


## Primal integral and statistical tests

### Checking missing logs

In [4]:
missing_logs = []
for instance_name, instance_log in instance_log_dict.items():
    for method_name, log_file_path in instance_log.items():
        if not log_file_path.exists():
            instance_path = log_file_path.parent.parent / log_file_path.parent.name.replace("_log", "") / log_file_path.name.replace(".log", "")
            missing_logs.append((instance_name, method_name, log_file_path, instance_path))

In [5]:
df_aux = pd.DataFrame(missing_logs, columns=["instance", "method", "log_file_path","instance_path"])
df_aux.groupby("method")["log_file_path"].count().sort_values(ascending=False)

Series([], Name: log_file_path, dtype: int64)

In [6]:
from utils import create_temp_file_list
if len(missing_logs)>0:
    file_paths = set()
    for instance_name, method_name, log_file_path, instance_path in missing_logs:
        file_paths.add(instance_path)
    file_paths = list(file_paths)
    aux = input("Enter step")
    output_filename = Path(base_path) / f"step_{aux}.txt"
    #print(len(file_paths),output_filename)
    aux = input(f"The file {output_filename} will be created with {len(file_paths)} instances. Press OK to confirm")
    if aux =="ok":
        create_temp_file_list(file_paths, output_filename=str(output_filename))
    else:
        raise Exception("File not created due to lack of confirmation")
else:
    print("You can continue")

You can continue


### Primal integral

In [7]:
from utils import get_all_logs_for_instance
if dataset== "MIS" or dataset== "MIS-mixed-train" or dataset == "CA":
    objective = "max"
elif dataset == "MVC" or dataset == "14_bounded_strongly_correlated":
    objective = "min"
else:
    raise ValueError(f"Dataset {dataset} not recognized.")
df_primal_integral,df = get_all_logs_for_instance(instance_log_dict, objective=objective)

In [8]:
df_primal_integral.groupby("method")["primal_integral"].describe()

,count,mean,std,min,25%,50%,75%,max
method,,,,,,,,
baseline,100.0,0.020951,0.029239,0.000000e+00,0.000005,0.000461,0.039991,0.099230
threshold_0.8005575056693743_alpha_0.4161451548840182,100.0,0.020628,0.029119,2.251634e-07,0.000008,0.000553,0.037246,0.099233


In [9]:
df_primal_gap = df.groupby(["method","instance"])["primal_gap"].min().reset_index().copy()

In [10]:
assert (df_primal_integral.groupby(["instance","method"]).count().max() == 1).all()

In [11]:
def method_renamed(x):
    if x in method_to_name:
        return method_to_name[x]
    elif x.startswith("conpas"):
        return "conpas"
    elif x.startswith("baseline"):
        return "gurobi"
    else:
        return x
def create_df_per_instance(df_param,metric):
    df_per_instance = df_param.copy()
    df_per_instance["method_renamed"] = df_per_instance["method"].apply(method_renamed)
    df_per_instance = df_per_instance[["method_renamed","instance",metric]].pivot_table(
        index="instance",
        columns="method_renamed",
        values=metric,
        aggfunc="first"
    ).add_prefix(f"{metric}_").reset_index()
    return df_per_instance

df_primal_gap_per_instance = create_df_per_instance(df_primal_gap,"primal_gap")
df_primal_integral_per_instance = create_df_per_instance(df_primal_integral,"primal_integral")

In [12]:
df_primal_integral_per_instance

method_renamed,instance,primal_integral_backpas,primal_integral_gurobi
0,trust_region_test_instance_0101_type14_n40000_...,0.067628,0.067626
1,trust_region_test_instance_0102_type14_n40000_...,0.000002,0.000002
2,trust_region_test_instance_0103_type14_n40000_...,0.067156,0.067198
3,trust_region_test_instance_0104_type14_n40000_...,0.035725,0.035743
4,trust_region_test_instance_0105_type14_n40000_...,0.000543,0.000543
...,...,...,...
95,trust_region_test_instance_0196_type14_n40000_...,0.000895,0.000896
96,trust_region_test_instance_0197_type14_n40000_...,0.000009,0.000010
97,trust_region_test_instance_0198_type14_n40000_...,0.000016,0.000019
98,trust_region_test_instance_0199_type14_n40000_...,0.000402,0.000402


In [13]:
summary = df_primal_integral_per_instance.describe().T.round(2)
summary.sort_values("mean",ascending=False)

,count,mean,std,min,25%,50%,75%,max
method_renamed,,,,,,,,
primal_integral_backpas,100.0,0.02,0.03,0.0,0.0,0.0,0.04,0.1
primal_integral_gurobi,100.0,0.02,0.03,0.0,0.0,0.0,0.04,0.1


### Statistical Tests

In [14]:
import pandas as pd
from scipy.stats import wilcoxon, ttest_rel



def run_tests(name_a, name_b, col_a, col_b, dataset_name,metric):
    alpha = 0.05

    # Wilcoxon tests
    w_stat_a_b, w_pval_a_b = wilcoxon(col_a, col_b, alternative='less')
    w_stat_b_a, w_pval_b_a = wilcoxon(col_a, col_b, alternative='greater')

    # Paired t-tests
    t_stat_a_b, t_pval_a_b = ttest_rel(col_a, col_b, alternative='less')
    t_stat_b_a, t_pval_b_a = ttest_rel(col_a, col_b, alternative='greater')

    # Directional conclusions
    wilcoxon_conclusion_a_b = f"{name_a} better" if w_pval_a_b < alpha else "not significant"
    wilcoxon_conclusion_b_a = f"{name_b} better" if w_pval_b_a < alpha else "not significant"

    ttest_conclusion_a_b = f"{name_a} better" if t_pval_a_b < alpha else "not significant"
    ttest_conclusion_b_a = f"{name_b} better" if t_pval_b_a < alpha else "not significant"

    return {
        "dataset": dataset_name,
        "metric":metric,
        "Method A": name_a,
        "Method B": name_b,

        # Wilcoxon values
        "wilcoxon_stat_A_better": w_stat_a_b,
        "wilcoxon_pvalue_A_better": w_pval_a_b,
        "wilcoxon_is_A_better": w_pval_a_b < alpha,
        "wilcoxon_conclusion_A_better": wilcoxon_conclusion_a_b,

        "wilcoxon_stat_B_better": w_stat_b_a,
        "wilcoxon_pvalue_B_better": w_pval_b_a,
        "wilcoxon_is_B_better": w_pval_b_a < alpha,
        "wilcoxon_conclusion_B_better": wilcoxon_conclusion_b_a,


        # t-test values
        "ttest_stat_A_better": t_stat_a_b,
        "ttest_pvalue_A_better": t_pval_a_b,
        "ttest_is_A_better": t_pval_a_b < alpha,
        "ttest_conclusion_A_better": ttest_conclusion_a_b,

        "ttest_stat_B_better": t_stat_b_a,
        "ttest_pvalue_B_better": t_pval_b_a,
        "ttest_is_B_better": t_pval_b_a < alpha,
        "ttest_conclusion_B_better": ttest_conclusion_b_a,
    }

def run_all_comparisons(df_per_instance, dataset_name,metric):
    results = []
    methods = ["backpas", "gurobi"]
    #foreach pair of methods run tests
    for i in range(len(methods)):
        for j in range(i+1, len(methods)):
            method_a = methods[i]
            method_b = methods[j]
            results.append(run_tests(
                method_a, method_b,
                df_per_instance[f'{metric}_{method_a}'],
                df_per_instance[f'{metric}_{method_b}'],
                dataset_name,
                metric
            ))
    return results
print("dataset:", dataset)
results_primal_integral = run_all_comparisons(df_primal_integral_per_instance, dataset_name=dataset, metric="primal_integral")
results_primal_gap = run_all_comparisons(df_primal_gap_per_instance, dataset_name=dataset, metric="primal_gap")
# Final dataframe
df_test_results = pd.DataFrame(results_primal_integral + results_primal_gap)
df_test_results.to_csv(f"test_results_{dataset}_statistical_ablation.csv", index=False)


dataset: 14_bounded_strongly_correlated


/home/redes/miniconda3/envs/backpas/lib/python3.10/site-packages/scipy/stats/_wilcoxon.py:172: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se


In [15]:
df_test_results[['dataset', 'metric', 'Method A', 'Method B', 'wilcoxon_conclusion_A_better', 'wilcoxon_conclusion_B_better', 'ttest_conclusion_A_better', 'ttest_conclusion_B_better']]

,dataset,metric,Method A,Method B,wilcoxon_conclusion_A_better,wilcoxon_conclusion_B_better,ttest_conclusion_A_better,ttest_conclusion_B_better
0,14_bounded_strongly_correlated,primal_integral,backpas,gurobi,not significant,not significant,not significant,not significant
1,14_bounded_strongly_correlated,primal_gap,backpas,gurobi,not significant,not significant,not significant,not significant
